In [ ]:
import logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

In [49]:
# from pathlib import Path

# # Resolves a relative path to an absolute path
# path = Path("example.txt").resolve()
# print(path) 


In [12]:
import logging
logger = logging.getLogger(__name__)

from typing import List, Any
from langchain_core.documents import Document
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
import unicodedata

def clean_text(text):
    # This replaces \u00a0 with a standard space and handles other oddities
    text = unicodedata.normalize("NFKD", text)
    # Optional: replace multiple spaces with a single space
    return " ".join(text.split())

# Apply this to your contract text before chunking
def load_all_document(data_dir:str)->List[Any]:
    """Load all documents from data directory"""
    data_path = Path(data_dir).resolve()
    logger.debug(f"data path: {data_path}")
    documents = []

    ## PDFs
    pdf_files = list(data_path.glob("**/*.pdf"))
    logger.debug(f"Found {len(pdf_files)} pdf files")
    for pdf_file in pdf_files:
        try:
            logger.debug(f"Loading pdf: {pdf_file}")
            loader = PyMuPDFLoader(str(pdf_file))
            docs = loader.load()
            for doc in docs:
                doc.page_content = clean_text(doc.page_content)
                doc.metadata["file_type"] = "pdf"
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata["contract_type"] = pdf_file.parent.name
            logger.debug(f"loader {len(docs)} docs from pdf: {pdf_file}")
            documents.extend(docs)
            logger.debug(f"loaded document: {pdf_file}")
        except Exception as e:
            logger.error(f"Failed to load {pdf_file}: {e}")
    logger.info(f"loaded {len(documents)}")
    return documents



In [13]:
docs = load_all_document("../data/test_3")

2026-04-08 15:42:28,127 | DEBUG | data path: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\test_3
2026-04-08 15:42:28,130 | DEBUG | Found 1 pdf files
2026-04-08 15:42:28,130 | DEBUG | Loading pdf: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\test_3\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF
2026-04-08 15:42:28,192 | DEBUG | loader 13 docs from pdf: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\test_3\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF
2026-04-08 15:42:28,196 | DEBUG | loaded document: C:\Users\elbou\Desktop\contractAssistant\contractAssistant\data\test_3\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF
2026-04-08 15:42:28,196 | INFO | loaded 13


In [14]:
docs

[Document(metadata={'producer': 'EVO HTML to PDF Converter 5.9', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\test_3\\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'file_path': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\test_3\\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf', 'source_file': 'ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'contract_type': 'test_3'}, page_content='Exhibit 10.3 EXHIBIT C SUPPORT AND MAINTENANCE AGREEMENT SUPPORT AND MAINTENANCE AGREEMENT dated as of April __, 2005 (the "Effective Date"), between On2 Technologies, Inc., a Delaware corporation ("On2"), and Wildform, Inc., a 

In [53]:
# print(f"Type of doc: {type(docs[0])}, Value: {docs[0]}")

### enrich metadata

In [15]:
import pandas as pd

## clean date fynction

def clean_date_human_display(raw_data:str)->str:
    # 1. Clean the string: Remove brackets and whitespace
    raw_date = str(raw_data).replace("[", "").replace("]", "").strip()
    # 2. Convert to a datetime object
    date_obj = pd.to_datetime(raw_date, format='mixed', errors='coerce')
    # 3. Format it back to your desired string style (MM/DD/YY)
    if pd.notnull(date_obj):
    # %m/%d/%y gives you 07/19/12. 
    # If you want 7/19/12 (no leading zero), use .strftime('%#m/%#d/%y') on Windows
        clean_date_str = date_obj.strftime('%m/%d/%y')
    else:
        clean_date_str = "Unknown" # Or keep it as None/nan
    return clean_date_str

## clean this for qdrant database
def clean_date_iso(raw_data:str):
    # 1. Clean the string: Remove brackets and whitespace
    raw_date = str(raw_data).replace("[", "").replace("]", "").strip()
    # 2. Convert to a datetime object
    date_obj = pd.to_datetime(raw_date, format='mixed', errors='coerce')
    # 3. Format it back to ISO format (YYYY-MM-DD)
    if pd.notnull(date_obj):
        clean_date_str = date_obj.strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        clean_date_str = "Unknown" 
    return clean_date_str

def clear_nan(raw_value):
    if pd.isna(raw_value):
        return "Unknown"
    else:
        return raw_value
def clear_brackets_and_nan(raw_value):
    ## ckear values with brackets and also the nan values
    if pd.isna(raw_value):
        return "Unknown"
    else:
        new_value =  str(raw_value).replace("[", "").replace("]", "").strip()
        if new_value == "":
            return "Unknown"
        return new_value
        


In [16]:
import re
import logging
import pandas as pd

logger = logging.getLogger(__name__)


def enrich_metadata(documents):

    logger.info(f"strat enriching metadata for {len(documents)} documents")
    data = pd.read_csv("../data/master_clauses.csv")

    for doc in documents:
        for index, row in data.iterrows():
            if doc.metadata.get("source_file") == row["Filename"]:
                #parties:
                parties = re.findall(r'([^;()]+)(?:\s*\(|$)', row["Parties-Answer"])
                parties = [party.strip() for party in parties if party.strip()] 
                party_1 = parties[0] if len(parties) > 0 else "Unknown"
                party_2 = parties[1] if len(parties) > 1 else "Unknown"
                doc.metadata.update({
                    "party_1":party_1,
                    "party_2":party_2,
                    "contract_type":row["Document Name-Answer"],
                    "agreement_date":clean_date_iso(row["Agreement Date-Answer"]),
                    "effective_date":clean_date_iso(row["Effective Date-Answer"]),
                    "expiration_date":clean_date_iso(row["Expiration Date-Answer"]),
                    "agreement_date_human_display":clean_date_human_display(row["Agreement Date-Answer"]),
                    "effective_date_human_display":clean_date_human_display(row["Effective Date-Answer"]),
                    "expiration_date_human_display":clean_date_human_display(row["Expiration Date-Answer"]),
                    "notice_period_to_terminate": clear_brackets_and_nan(row["Notice Period To Terminate Renewal- Answer"]),
                    "renewl_term":clear_nan(row["Renewal Term-Answer"]),
                    "governing_law":row["Governing Law-Answer"]

                })
    logger.info(f"metadata enriched successfully")
    return documents


In [17]:
docs = enrich_metadata(docs)

2026-04-08 15:42:55,072 | INFO | strat enriching metadata for 13 documents
2026-04-08 15:42:56,097 | INFO | metadata enriched successfully


In [18]:
docs

[Document(metadata={'producer': 'EVO HTML to PDF Converter 5.9', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\test_3\\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'file_path': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\test_3\\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf', 'source_file': 'ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'contract_type': 'SUPPORT AND MAINTENANCE AGREEMENT', 'party_1': 'On2 Technologies, Inc.', 'party_2': 'Wildform, Inc.', 'agreement_date': '2005-04-01T00:00:00Z', 'effective_date': '2005-04-01T00:00:00Z', 'expiration_date': '2006-10-01T00:00:00Z', 'agreement_date_human_d

### chunking

In [119]:
import logging
import re
from typing import List
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

logger = logging.getLogger(__name__)

SECTION_PATTERN = re.compile(r"\n\s*(\d+)\.\s+[A-Z][^\n]+")


def split_by_sections(text: str):
    """
    Split contract text by legal section headers like:
    1. Services
    2. Compensation
    """

    matches = list(SECTION_PATTERN.finditer(text))
    if not matches:
        return [text]
    sections = []
    for i, match in enumerate(matches):
        header_info = f"Section {match.group(0)}:\n"

        start = match.start()
        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)
        section_content = text[start:end].strip()
        sections.append(f"{header_info}---------------------------------{section_content}")
    return sections


def chunk_contract_documents(documents: List[Document]) -> List[Document]:
    """
    Chunk contracts using section-aware chunking,
    then semantic chunking for large sections.
    """
    ## add model_kwargs and encode_kwargs to force using GPU
    # model_kwargs = {"device":"cuda"}
    # encode_kwargs = {"normalize_embeddings": True}
    # embeddings = HuggingFaceEmbeddings(
    #     model_name = "BAAI/bge-large-en-v1.5",
    #     model_kwargs = model_kwargs,
    #     encode_kwargs = encode_kwargs
        
    # )

    # semantic_chunker = SemanticChunker(
    #     embeddings,
    #     breakpoint_threshold_type="percentile",
    #     breakpoint_threshold_amount=90,
    # )
    
    logger.info(f"initialise embedding model for Semantic Chunker ")
    all_chunks = []
    logger.info("start chunking ...")
    for doc in documents:
        text = doc.page_content
        # split by contract sections
        sections = split_by_sections(text)
        for section in sections:
            if len(section) < 2500: 
                all_chunks.append(Document(page_content=section, metadata=doc.metadata.copy()))
        else:
            # For massive sections, split with OVERLAP so context isn't lost at the cut
            from langchain_text_splitters import RecursiveCharacterTextSplitter
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1200,
                chunk_overlap=200, # This prevents the "response not found" at the edges
                add_start_index=True
            )
            splits = text_splitter.split_text(section)
            for split in splits:
                all_chunks.append(Document(page_content=split, metadata=doc.metadata.copy()))


    all_chunks =  [doc for doc in all_chunks if len(doc.page_content) > 25]
    logger.info(f"Created {len(all_chunks)} chunks")
    return all_chunks


In [120]:
chunks = chunk_contract_documents(docs)

2026-04-08 17:50:42,641 | INFO | initialise embedding model for Semantic Chunker 
2026-04-08 17:50:42,641 | INFO | start chunking ...
2026-04-08 17:50:42,654 | INFO | Created 61 chunks


In [121]:
chunks

[Document(metadata={'producer': 'EVO HTML to PDF Converter 5.9', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\test_3\\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'file_path': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\test_3\\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf', 'source_file': 'ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'contract_type': 'SUPPORT AND MAINTENANCE AGREEMENT', 'party_1': 'On2 Technologies, Inc.', 'party_2': 'Wildform, Inc.', 'agreement_date': '2005-04-01T00:00:00Z', 'effective_date': '2005-04-01T00:00:00Z', 'expiration_date': '2006-10-01T00:00:00Z', 'agreement_date_human_d

In [123]:
chunks[20:]

[Document(metadata={'producer': 'EVO HTML to PDF Converter 5.9', 'creator': '', 'creationdate': '', 'source': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\test_3\\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'file_path': 'C:\\Users\\elbou\\Desktop\\contractAssistant\\contractAssistant\\data\\test_3\\ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'total_pages': 13, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 3, 'file_type': 'pdf', 'source_file': 'ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF', 'contract_type': 'SUPPORT AND MAINTENANCE AGREEMENT', 'party_1': 'On2 Technologies, Inc.', 'party_2': 'Wildform, Inc.', 'agreement_date': '2005-04-01T00:00:00Z', 'effective_date': '2005-04-01T00:00:00Z', 'expiration_date': '2006-10-01T00:00:00Z', 'agreement_date_human_d

In [ ]:
# # delete collection

# from qdrant_client import QdrantClient

# client = QdrantClient(host="localhost", port=6333)
# client.delete_collection(collection_name="contracts")  # deletes old collection

2026-04-08 18:13:55,440 | DEBUG | connect_tcp.started host='localhost' port=6333 local_address=None timeout=5.0 socket_options=None
2026-04-08 18:13:55,440 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000021D34121950>
2026-04-08 18:13:55,440 | DEBUG | send_request_headers.started request=<Request [b'DELETE']>
2026-04-08 18:13:55,440 | DEBUG | send_request_headers.complete
2026-04-08 18:13:55,440 | DEBUG | send_request_body.started request=<Request [b'DELETE']>
2026-04-08 18:13:55,440 | DEBUG | send_request_body.complete
2026-04-08 18:13:55,448 | DEBUG | receive_response_headers.started request=<Request [b'DELETE']>
2026-04-08 18:13:55,625 | DEBUG | connect_tcp.started host='localhost' port=6333 local_address=None timeout=5 socket_options=None
2026-04-08 18:13:55,641 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000021CAD618710>
2026-04-08 18:13:55,641 | DEBUG | send_request_headers.started re

True

### store in the qdrant db

In [24]:
## creating data
import logging
import hashlib
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models

logger = logging.getLogger(__name__)


## embeed then store
class QdrantStore:

    def __init__(
        self,
        dense_model_name: str = "BAAI/bge-large-en-v1.5",
        sparse_model_name: str = "Qdrant/bm25",
    ):
        self.dense_model_name = dense_model_name
        self.sparse_model_name = sparse_model_name
        ## add model_kwargs={'device': 'cuda'} for GPU Enabling
        self.dense_model = SentenceTransformer(self.dense_model_name,device="cuda")
        ## add providers=["CUDAExecutionProvider"] for GPU Enabling
        self.sparse_model = SparseTextEmbedding(model_name=self.sparse_model_name, providers=["CUDAExecutionProvider"])
        self.client = QdrantClient(host="localhost", port=6333, timeout=60)
        self.collection_name = "contracts_v1"
        self.creat_collection()

    def creat_collection(self):
        if not self.client.collection_exists(collection_name=self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config={
                    "dense": models.VectorParams(
                        size=1024, distance=models.Distance.COSINE
                    )
                },
                sparse_vectors_config={
                    "sparse": models.SparseVectorParams(
                        index=models.SparseIndexParams(on_disk=False)
                    )
                },
                timeout=60
            )
    ## generate IDs
    def generate_doc_id(self, source: str, content: str):
        unique_string = f"{source}:{content}"
        return hashlib.sha256(unique_string.encode()).hexdigest()[:32]

    def embedde_chunks_and_store(self, chunks):

        logger.info(f"generating embedding {len(chunks)} chunks ...")

        texts = [chunk.page_content for chunk in chunks]

        dense_embeddings = self.dense_model.encode(
            texts, normalize_embeddings=True, batch_size=16, show_progress_bar=True
        )
        sparse_embeddings = list(self.sparse_model.embed(texts))

        # Focuses on Semantic Direction: By removing the effect of vector magnitude, normalization ensures that similarity is based on semantic meaning rather than frequency or length.
        logger.info(f"embeddings shape: {dense_embeddings.shape}")

        points = []
        batch_size = 64

        for chunk, dense_embedding, sparse_embedding in zip(
            chunks, dense_embeddings, sparse_embeddings
        ):
            ## generating ID
            source = chunk.metadata.get("source_file","uknown")
            doc_id = self.generate_doc_id(source,chunk.page_content)

            ## getting general context
            contract_type =chunk.metadata.get("contract_type","Legal Document")
            parties = f"{chunk.metadata.get('party_1','')} and {chunk.metadata.get('party_2','')}"
            
            #context header
            contract_info = f"Type: {chunk.metadata.get('contract_type', 'Legal')}"
            points.append(
                models.PointStruct(
                    # id=uuid.uuid4(),
                    id=doc_id,
                    vector={
                        "dense": dense_embedding.tolist(),
                        "sparse": models.SparseVector(
                            indices=sparse_embedding.indices,
                            values=sparse_embedding.values,
                        ),
                    },
                    payload={
                        # "text": f"Document: {contract_type} between {parties}, \n Context: {chunk.page_content}",

                        "text": chunk.page_content,
                        "context_header": f"{source} from {contract_info}",
                        **chunk.metadata,
                    },
                )
            )
            if len(points) > batch_size:
                self.client.upsert(collection_name=self.collection_name, points=points)
                logger.info(f"upsertted {len(points)}..")
                points = []
        if points:
            self.client.upsert(collection_name=self.collection_name, points=points)
            logger.info(f"upsertted final {len(points)} points.")

        logger.info("chunks stored in Qdrant.")


In [25]:
store = QdrantStore()
store.embedde_chunks_and_store(chunks)

2026-04-08 15:45:11,936 | INFO | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-04-08 15:45:12,367 | DEBUG | connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-04-08 15:45:12,416 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000021CAD3C3010>
2026-04-08 15:45:12,416 | DEBUG | start_tls.started ssl_context=<ssl.SSLContext object at 0x0000021CAD2BA690> server_hostname='huggingface.co' timeout=10
2026-04-08 15:45:12,493 | DEBUG | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000021CAD130A50>
2026-04-08 15:45:12,495 | DEBUG | send_request_headers.started request=<Request [b'HEAD']>
2026-04-08 15:45:12,496 | DEBUG | send_request_headers.complete
2026-04-08 15:45:12,496 | DEBUG | send_request_body.started request=<Request [b'HEAD']>
2026-04-08 15:45:12,496 | DEBUG | send_request_body.complete
2026-04-08 15:45:12,496 | DEBUG | receive_respon

### rewrite query

In [98]:

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
import os 

load_dotenv()

class QueryRewriting:
    def __init__(self, model_name:str="llama-3.3-70b-versatile"): #
        self.model_name = model_name

        groq_api_key = os.getenv("GROQ_API_KEY")
        if not groq_api_key:
            raise ValueError("GROQ_API_KEY not found in environment variables")
        self.llm = ChatGroq(groq_api_key = groq_api_key, model_name = self.model_name, temperature=0)

        self.prompt = PromptTemplate(
            input_variables=["original_query"],
          template="""You are an expert legal query rewriting and expansion assistant.

            Your task is to rewrite the user's query into aretrieval-optimized search query for legal contracts.

            Rules:
            - Preserve the original legal intent exactly (DO NOT add new concepts)
            - Expand the query by at most five words without changing the legal meaning or terminology.            
            - Avoid document-specific names, metadata, or filters
            - Output a single query

            Original query: {original_query}

            Rewritten query:"""
        )
    
        self.chain = self.prompt | self.llm
        print(f"[INFO] Groq LLM initialized {self.model_name}")

    def rewrite_query(self,original_query:str):
        if not original_query.strip():
            raise ValueError("original query cannot be empty")
        response = self.chain.invoke({"original_query": original_query})
        return response.content.strip()


In [99]:
original_query = "Identify any 'Exclusivity' rights granted in 'ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF'."
rewriter = QueryRewriting()
rewriten = rewriter.rewrite_query(original_query)

2026-04-08 17:08:50,132 | DEBUG | Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-55895c45-3ba0-44ef-b8b7-55e0985e7cb9', 'json_data': {'messages': [{'role': 'user', 'content': "You are an expert legal query rewriting and expansion assistant.\n\n            Your task is to rewrite the user's query into aretrieval-optimized search query for legal contracts.\n\n            Rules:\n            - Preserve the original legal intent exactly (DO NOT add new concepts)\n            - Expand the query by at most five words without changing the legal meaning or terminology.            \n            - Avoid document-specific names, metadata, or filters\n            - Output a single query\n\n            Original query: Identify any 'Exclusivity' rights granted in 'ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF'.\n\n            Rewritten query:"}], 'model': 'llama-3.3-70b-versatile', 'n'

[INFO] Groq LLM initialized llama-3.3-70b-versatile


2026-04-08 17:08:50,515 | DEBUG | receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 08 Apr 2026 16:08:50 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'Cache-Control', b'private, max-age=0, no-store, no-cache, must-revalidate'), (b'Server', b'cloudflare'), (b'vary', b'Origin'), (b'x-groq-region', b'fra'), (b'x-ratelimit-limit-requests', b'1000'), (b'x-ratelimit-limit-tokens', b'12000'), (b'x-ratelimit-remaining-requests', b'996'), (b'x-ratelimit-remaining-tokens', b'11807'), (b'x-ratelimit-reset-requests', b'5m45.6s'), (b'x-ratelimit-reset-tokens', b'965ms'), (b'x-request-id', b'req_01knpxn0sgf9zak01xhddfep5t'), (b'set-cookie', b'__cf_bm=SfFlAwQC_F59Sq7iqMJQl9lYMbB0M2gJ_Cp9cNO1ok8-1775664530.2095985-1.0.1.1-hMgaTpRQm15xNxEiFZEEDYfQXd7G51.T8Vp5Qe0A2JYprdMhsRvQPob0J68Z_d4sKDGYm7e1ka4ryu_uls9QxGm8pbVzepAAj6RYlsaIERiLL8sGIegLY_soclBtT_yX; HttpOnly; Secure; Path=/; Domain=groq.com

In [100]:
rewriten
#'Identify the parties executing the agreement in the contract document.'

"Identify any 'Exclusivity' rights granted in the agreement."

### filters

### Query constructor

In [ ]:
from langchain_classic.chains.query_constructor.base import load_query_constructor_runnable, load_query_constructor_chain
from langchain_community.query_constructors.qdrant import QdrantTranslator
from langchain_classic.chains.query_constructor.ir import Comparison, Operation
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from qdrant_client import models
import os
from langchain_groq import ChatGroq
def flatten_metadata_values(node):
    """
    Recursively finds dictionaries like {'date': '...'} in the AST 
    and replaces them with the actual string value.
    """
    if isinstance(node, Comparison):
        # If the value is that problematic dictionary, grab just the date string
        if isinstance(node.value, dict) and 'date' in node.value:
            node.value = node.value['date']
            
    elif isinstance(node, Operation):
        # Recursively check all arguments in AND/OR operations
        for arg in node.arguments:
            flatten_metadata_values(arg)
    return node
def remove_content_prefix(node):
    """
    Recursively removes 'content.' from any attribute names in the AST.
    """
    if hasattr(node, 'attribute') and node.attribute.startswith("content."):
        node.attribute = node.attribute.replace("content.", "")
            
    if hasattr(node, 'arguments'):
        for arg in node.arguments:
            remove_content_prefix(arg)
    return node
def clean_qdrant_filters(filter_obj):
    if filter_obj is None:
        return None

    # Helper to clean a single condition (the core logic)
    def clean_single_condition(condition):
        if isinstance(condition, models.FieldCondition):
            new_key = condition.key.replace("content.", "")
            if new_key.startswith("."):
                new_key = new_key[1:]
            condition.key = new_key
        return condition

    # 1. If it's a direct FieldCondition (single filter)
    if isinstance(filter_obj, models.FieldCondition):
        return clean_single_condition(filter_obj)

    # 2. If it's a Filter object (nested/multiple filters)
    if isinstance(filter_obj, models.Filter):
        if filter_obj.must:
            for c in filter_obj.must: clean_single_condition(c)
        if filter_obj.should:
            for c in filter_obj.should: clean_single_condition(c)
        if filter_obj.must_not:
            for c in filter_obj.must_not: clean_single_condition(c)
        
        # Check for nested filters inside this filter
        # (Recursive calls if you have nested AND/OR)
        for attr in ['must', 'should', 'must_not']:
            cond_list = getattr(filter_obj, attr)
            if cond_list:
                for c in cond_list:
                    if isinstance(c, models.Filter):
                        clean_qdrant_filters(c)
                        
    return filter_obj
def get_filter_from_query(query: str): 
    metadata_info = [
        AttributeInfo(
            name="source_file",
            description="The name of the document. Use this to filter by name of document",
            type="string"
        ),
        AttributeInfo(
            name="party_1",
            description="The name of the first legal entity or company mentioned in the contract. Use this to filter by the primary party or signer.",
            type="string"
        ),
        AttributeInfo(
            name="party_2",
            description="The name of the second legal entity or counterparty in the contract. Use this to filter for the second participant in the agreement.",
            type="string"
        ),
        AttributeInfo(
            name="contract_type",
            description="The category of the agreement, such as 'Franchise Agreement', 'NDA', 'Service Agreement', or 'Lease'. Use this when the user specifies a document type.",
            type="string"
        ),
        AttributeInfo(
            name="agreement_date",
            description="The date the contract was signed or formally created. Use for questions about when a contract was made.",
            type="string"
        ),
        AttributeInfo(
            name="effective_date",
            description="The official start date of the contract's obligations. Use for questions about when an agreement becomes active.",
            type="string"
        ),
        AttributeInfo(
            name="expiration_date",
            description="The date the contract ends or becomes invalid. Use for questions about renewals, terminations, or end dates.",
            type="string"
        ),
        AttributeInfo(
            name="governing_law",
            description="The legal jurisdiction or state laws that apply to the contract (e.g., 'California', 'New York', 'Morocco'). Use this when the user mentions a specific location or law.",
            type="string"
        ),
    ]

    document_content_description = "Detailed clauses and legal text from corporate contracts"

    groq_api_key = os.getenv("GROQ_API_KEY")
    
    # Enable JSON mode for reliable structured output
    llm = ChatGroq(
        model_name="openai/gpt-oss-120b", #llama-3.3-70b-versatile
        groq_api_key=groq_api_key,
        temperature=0,
        # model_kwargs={
        #     "response_format": {"type": "json_object"}  # Force JSON output
        # }
    )
    constractor_chain = load_query_constructor_runnable(
        llm,
        document_content_description,
        metadata_info
    )
    result = constractor_chain.invoke({"query": query})
    # print(f"result: {result}")
    translator = QdrantTranslator(metadata_key="")
    
    if result.filter:
        try:
            # 1. Clean the AST first
            clean_filter = flatten_metadata_values(result.filter)
            clean_filter = remove_content_prefix(clean_filter) # Use your helper
            
            # 2. Check the type of the filter to use the correct visitor method
            if isinstance(clean_filter, Comparison):
                qdrant_filter = translator.visit_comparison(clean_filter)
            elif isinstance(clean_filter, Operation):
                qdrant_filter = translator.visit_operation(clean_filter)
            else:
                # Fallback for unexpected types
                qdrant_filter = None
            
            # 3. Final cleaning for Qdrant-specific FieldConditions
            if qdrant_filter:
                qdrant_filter = clean_qdrant_filters(qdrant_filter)
            if isinstance(qdrant_filter, models.FieldCondition):
                qdrant_filter = models.Filter(must=[qdrant_filter])
                
            return qdrant_filter
                
        except Exception as e:
            print(f"Translation failed: {e}")
            qdrant_filter = None
    else:
        qdrant_filter = None

    return qdrant_filter


In [43]:
filters = get_filter_from_query(original_query)

2026-04-08 16:25:57,440 | DEBUG | Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-f5b1a09a-415a-4ed2-8247-8f0f3dbf7d06', 'json_data': {'messages': [{'role': 'user', 'content': 'Your goal is to structure the user\'s query to match the request schema provided below.\n\n<< Structured Request Schema >>\nWhen responding use a markdown code snippet with a JSON object formatted in the following schema:\n\n```json\n{\n    "query": string \\ text string to compare to document contents\n    "filter": string \\ logical condition statement for filtering documents\n}\n```\n\nThe query string should contain only text that is expected to match the contents of documents. Any conditions in the filter should not be mentioned in the query as well.\n\nA logical condition statement is composed of one or more comparison and logical operation statements.\n\nA comparison statement takes the form: `comp(attr, val)`:\n- `comp` (

result: query='Exclusivity' filter=Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='source_file', value='ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF') limit=None


In [44]:
print(filters)

should=None min_should=None must=[FieldCondition(key='source_file', match=MatchValue(value='ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)] must_not=None


### hybrid search

In [45]:

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, FusionQuery, Fusion, Filter
from fastembed import SparseTextEmbedding
from typing import Optional
from dataclasses import dataclass


# result schema
@dataclass
class SearchResult:
    chunk_id:str
    context_header:str
    metadata:dict 


class HybridSearch:
    def __init__(self, dense_model_name:str= "BAAI/bge-large-en-v1.5", sparse_model_name:str="Qdrant/bm25"):
        self.dense_model_name = dense_model_name
        self.dense_model = SentenceTransformer(self.dense_model_name, device="cuda")
        self.sparse_model_name = sparse_model_name
        self.sparse_model = SparseTextEmbedding(self.sparse_model_name)

        self.client = QdrantClient(host="localhost", port=6333)
        self.collection_name = "contracts_v1"
    def hybrid_search_with_rrf(self, query:str, filters:Optional[Filter]=None):
        """Perform hybrid search using Reciprocal Rank Fusion"""
        # rrf is a method for merging and ranking results from multiple search techniques
        query_dense_embedding = self.dense_model.encode(query,normalize_embeddings=True,batch_size=16)
        query_sparse_embeding = list(self.sparse_model.embed(query))[0]

        query_sparse_vector = models.SparseVector(
            indices=query_sparse_embeding.indices,
            values=query_sparse_embeding.values
        )
        ## hybrifd search
        results = self.client.query_points(
            collection_name=self.collection_name,
            prefetch=[
                Prefetch(
                    query=query_dense_embedding,
                    using="dense",
                    limit=10,
                    filter=filters
                ),
                Prefetch(
                    query=query_sparse_vector,
                    using="sparse",
                    limit=10,
                    filter=filters
                )
            ],
            query= FusionQuery(fusion=Fusion.RRF)
        )
        return self.hybrid_search_points_to_results(results.points)
    
    def hybrid_search_points_to_results(self, points):

        return [
            SearchResult(
                chunk_id=str(point.id),
                context_header = str(point.payload.get("context_header","")),
                metadata=point.payload
            )
            for point in points
        ]



In [101]:
hybrid_search = HybridSearch()
results = hybrid_search.hybrid_search_with_rrf(query=rewriten,filters=filters)

2026-04-08 17:09:07,064 | INFO | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-04-08 17:09:07,067 | DEBUG | close.started
2026-04-08 17:09:07,068 | DEBUG | close.complete
2026-04-08 17:09:07,068 | DEBUG | connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-04-08 17:09:07,120 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000021D34087BD0>
2026-04-08 17:09:07,121 | DEBUG | start_tls.started ssl_context=<ssl.SSLContext object at 0x0000021CAD2BA690> server_hostname='huggingface.co' timeout=10
2026-04-08 17:09:07,165 | DEBUG | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000021D82F51490>
2026-04-08 17:09:07,166 | DEBUG | send_request_headers.started request=<Request [b'HEAD']>
2026-04-08 17:09:07,166 | DEBUG | send_request_headers.complete
2026-04-08 17:09:07,166 | DEBUG | send_request_body.started request=<Request [b'HEAD']>
2026-04-08 1

In [47]:
results

[SearchResult(chunk_id='c5413e6f-1abc-5009-fd27-e9377ff5969c', context_header='ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF from Type: SUPPORT AND MAINTENANCE AGREEMENT', metadata={'text': 'this Agreement or the transactions contemplated hereby shall be instituted by a party in a Federal or state court sitting in the jurisdiction and venue of the other party, which shall be the exclusive jurisdiction and venue of said legal proceedings and each party hereto waives any objection which such party may now or hereafter have to the laying of venue of any such action, suit or proceeding, and irrevocably submits to the jurisdiction of any such court in any such action, suit or proceeding. Any and all service of process and any other notice in any such action, suit or proceeding shall be effective against such party (or the subsidiary of such party) when transmitted in accordance with Section 10.8. Nothing contained herein shall be deemed to affect the right of 

### rerank

In [73]:
import logging
from dataclasses import dataclass
import os 
from dotenv import load_dotenv
import cohere
from typing import Optional

load_dotenv()
logger = logging.getLogger(__name__)

@dataclass
class RerankedResult:
    chunk_id: str
    # text: str
    rerank_score: float
    metadata: dict
    original_rank: int
 

class Reranker:
    # rerank-v3.5 / rerank-v4.0
    #rerank-english-v3.0
    def __init__(self, model_name:str = "rerank-v3.5"):
        self.model_name = model_name
        cohere_api_key = os.getenv("COHERE_API_KEY")
        if not cohere_api_key:
            raise ValueError("cohere api key not found in the .env file")
        self.client = cohere.Client(cohere_api_key)
        logger.info(f"Reranker initialized - model: {model_name}")

    def rerank(self, query: str, results: list, top_n:int=8)->list:
        """
        Rerank hybrid search results using Cohere API. 
        Returns:
            list of RerankedResult sorted by rerank_score descending
        """
        if not results:
            return []
        documents = [r.metadata.get("text","") for r in results] # r.page_content for r in results

        response = self.client.rerank(
            model = self.model_name,
            query = query,
            documents = documents,
            top_n = top_n,
            return_documents = False
        )

        reranked = []
        for hit in response.results:
            original = results[hit.index]
            reranked.append(RerankedResult(
                chunk_id=original.chunk_id,
                rerank_score=hit.relevance_score,
                metadata=original.metadata,
                original_rank=hit.index + 1,
            ))
        logger.info(f"Cohere reranked {len(results)} → top {len(reranked)} results")

        return reranked




In [102]:
reranker = Reranker()
results = reranker.rerank(rewriten,results)

2026-04-08 17:09:16,700 | INFO | Reranker initialized - model: rerank-v3.5
2026-04-08 17:09:16,701 | DEBUG | connect_tcp.started host='api.cohere.com' port=443 local_address=None timeout=300 socket_options=None
2026-04-08 17:09:16,935 | DEBUG | connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000021CAD2EF910>
2026-04-08 17:09:16,935 | DEBUG | start_tls.started ssl_context=<ssl.SSLContext object at 0x0000021D34001EB0> server_hostname='api.cohere.com' timeout=300
2026-04-08 17:09:16,973 | DEBUG | start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x0000021CAD61B1D0>
2026-04-08 17:09:16,974 | DEBUG | send_request_headers.started request=<Request [b'POST']>
2026-04-08 17:09:16,974 | DEBUG | send_request_headers.complete
2026-04-08 17:09:16,975 | DEBUG | send_request_body.started request=<Request [b'POST']>
2026-04-08 17:09:16,975 | DEBUG | send_request_body.complete
2026-04-08 17:09:16,976 | DEBUG | receive_response_headers.start

In [50]:
results

[RerankedResult(chunk_id='1384e7a0-b1a1-96f8-abf9-03381660a006', rerank_score=0.4581311, metadata={'text': 'shall be enforceable to the fullest extent permitted by law. The invalidity and/or unenforceability in whole or in part of any provision of this Agreement shall not render invalid or unenforceable any other provision of this Agreement, which instead will remain in full force and effect. 10.6 Entire Agreement. This Agreement constitutes the entire understanding between the parties regarding to specific subject matter covered herein. This Agreement supersedes any and all prior written or verbal contracts or understandings between the parties hereto and neither party shall be bound by any statements or representations made by either party not embodied in this Agreement. No provisions herein contained shall be waived, modified or altered, except by an instrument in writing, duly executed by the parties hereto. 10.7 Governing Law; Forum. This Agreement shall be governed by and constru

## Generation

## question answering

In [109]:
from typing import List, Dict

SYSTEM_PROMPT = """You are a Senior Legal Counsel specializing in contract audit and compliance.
Your goal is to extract precise, verifiable information from contract excerpts.

CORE OPERATIONAL PROTOCOLS:

1. GROUNDING:
Answer ONLY using the provided text. Do not use outside knowledge or assumptions.

2. CITATIONS:
For every fact you provide, you MUST include a citation (e.g., [Source: Section 4.2]).

3. MISSING VS NON-EXISTENT:
- If relevant sections are retrieved but do NOT contain the requested concept, state clearly:
  "The agreement does not grant or define [topic]."
- If no relevant information is found at all, state:
  "The provided documents do not contain information regarding [topic]."

4. PRECISION:
Preserve the exact wording of defined legal terms (e.g., "Effective Date", "Force Majeure").

5. STEP-BY-STEP REASONING:
- Identify the most relevant section(s)
- Verify whether the concept is explicitly stated
- Extract and report the answer

6. LEGAL DISAMBIGUATION:
Do NOT confuse similar terms. For example:
- "exclusive jurisdiction" ≠ "exclusivity rights"
- Only report business/legal rights relevant to the query

7. NO OVERGENERALIZATION:
Do not infer rights, obligations, or clauses that are not explicitly stated.

8. Never paraphrase legal clauses in a way that introduces new meaning

OUTPUT:
Provide a concise, legally precise answer with citations only.
"""

def build_user_prompt(query: str, chunks: List[Dict]) -> str:

    texts = [chunk.get("text", "") for chunk in chunks]
    context = "\n\n".join(texts)
    return f"""### CONTRACT EXCERPTS:
        {context}

        ### INSTRUCTIONS:
        Analyze the excerpts above to answer the following question. Include citations to the Document Name and Chunk number where possible.

        QUESTION: {query}
        ANSWER:"""


## llm client

In [110]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
import json

import os
from dotenv import load_dotenv
from torch import chunk


load_dotenv()


class LLMClient:

    def __init__(self, model_name: str = "nvidia/nemotron-3-super-120b-a12b:free"): #llama-3.3-70b-versatile
        """Groq LLM client for contract question answering."""
        self.model_name = model_name
        openrouter_key = os.getenv("OPENROUTER_API_KEY")
        if not openrouter_key:
            raise ValueError("can't find the openrouter_key in .env")
        self.llm = ChatOpenAI(
            model_name= self.model_name,
            openai_api_key=openrouter_key,
            openai_api_base="https://openrouter.ai/api/v1"
        )


        print(f"[INFO] LlmClient initialized - model {self.model_name}")

    def generate_response(self, query: str, chunks):
        USER_PROMPT = build_user_prompt(query, chunks)

        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=USER_PROMPT),
        ]
        try:
            answer = self.llm.invoke(messages)
        except Exception as e:
            raise RuntimeError(f"openrouter call failed: {e}") from e
        print(f"llm answer: {answer}")
        sources = [
            {
                "file_name": chunk.get("source_file", ""),
                "file_type": chunk.get("file_type", ""),
                "page": chunk.get("page", ""),
                "contract_type": chunk.get("contract_type", ""),
                "agreement_date": chunk.get("agreement_date", ""),
                "effective_date": chunk.get("effective_date", ""),
                "expiration_date": chunk.get("expiration_date", ""),
                "agreement_date_human_display": chunk.get(
                    "agreement_date_human_display", ""
                ),
                "effective_date_human_display": chunk.get(
                    "effective_date_human_display", ""
                ),
                "expiration_date_human_display": chunk.get(
                    "expiration_date_human_display", ""
                ),
                "party_1": chunk.get("party_1", ""),
                "party_2": chunk.get("party_2", ""),
                "notice_period_to_terminate": chunk.get(
                    "notice_period_to_terminate", ""
                ),
                "renewl_term": chunk.get("renewl_term", ""),
                "governing_law": chunk.get("governing_law", ""),
                "preview": chunk["text"][0:200] + "...",
            }
            for chunk in chunks
        ]

        results = {
            "answer": answer.content,
            "sources": sources,
        }
        return self.format_result(results)
        # return results

    @staticmethod
    def format_result(results):
    # results is likely a dict containing a list under "sources"
    # or just a list itself. Adjust accordingly:
        sources_input = results.get("sources", [])
        
        sources_map = {}

        for source in sources_input:
            filename = source.get("file_name", "unknown")
            
            if filename not in sources_map:
                # First time seeing this file → create the entry
                sources_map[filename] = {
                    "filename": filename,
                    "file_type": source.get("file_type", ""),
                    "page": source.get("page", ""),
                    "contract_type": source.get("contract_type", ""),
                    "agreement_date": source.get("agreement_date", ""),
                    "effective_date": source.get("effective_date", ""),
                    "expiration_date": source.get("expiration_date", ""),
                    "agreement_date_human_display": source.get("agreement_date_human_display", ""),
                    "effective_date_human_display": source.get("effective_date_human_display", ""),
                    "expiration_date_human_display": source.get("expiration_date_human_display", ""),
                    "party_1": source.get("party_1", ""),
                    "party_2": source.get("party_2", ""),
                    "pages": [source["page"]] if source.get("page") else [],
                    "notice_period_to_terminate": source.get("notice_period_to_terminate", ""),
                    "renewl_term": source.get("renewl_term", ""),
                    "governing_law": source.get("governing_law", ""),
                    "preview": source.get("preview", ""),
                }
            else:
                # Already seen this file → update the pages list
                existing = sources_map[filename]
                page = source.get("page")
                if page and page not in existing["pages"]:
                    existing["pages"].append(page)

        return {
            "answer": results["answer"],
            "sources": list(sources_map.values()),
            }

    # ── Chunk formatter ───────────────────────────────────────────────────────
    @staticmethod
    def reranked_to_chunks(reranked_results: list) -> list[dict]:
        """
        Convert RerankedResult objects from reranker.py into
        the simple dict format expected by build_user_prompt().

        Args:
            reranked_results : list of RerankedResult from Reranker.rerank()

        Returns:
            list of dicts with keys: text, file_name, section_title, chunk_id
        """
        return [
            {
                **r.metadata,
                "chunk_id": r.chunk_id,
                "text": r.metadata.get("text", ""),
                "file_name": r.metadata.get("source", "unknown"),
                "original_score": r.original_rank,
            }
            for r in reranked_results
        ]

    # if __name__ == "__main__":


#     llm_client = LLMClient()
#     reranker = Reranker()
#     query_rewriter = QueryRewriting()
#     hybrid_search = HybridSearch()
#     original_query = "what is bla bla bla ?"

#     qdrant_filters = get_filter_from_query(original_query)
#     rewrited_query = query_rewriter.rewrite_query(original_query)
#     results = hybrid_search.hybrid_search_with_rrf(rewrited_query,fliters=qdrant_filters)
#     reranked_results = reranker.rerank(rewrited_query, results)
#     chunks = llm_client.reranked_to_chunks(reranked_results)

#     answer = llm_client.generate_response(rewrited_query,chunks)

#     print(answer)

In [111]:
llm = LLMClient()

[INFO] LlmClient initialized - model nvidia/nemotron-3-super-120b-a12b:free


In [112]:
chunks = llm.reranked_to_chunks(results)

In [113]:
# chunks

In [114]:
answer = llm.generate_response(rewriten,chunks)

2026-04-08 17:22:53,697 | DEBUG | Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-f169231c-53d0-4bb5-a49a-1a500717d579', 'content': None, 'json_data': {'messages': [{'content': 'You are a Senior Legal Counsel specializing in contract audit and compliance.\nYour goal is to extract precise, verifiable information from contract excerpts.\n\nCORE OPERATIONAL PROTOCOLS:\n\n1. GROUNDING:\nAnswer ONLY using the provided text. Do not use outside knowledge or assumptions.\n\n2. CITATIONS:\nFor every fact you provide, you MUST include a citation (e.g., [Source: Section 4.2]).\n\n3. MISSING VS NON-EXISTENT:\n- If relevant sections are retrieved but do NOT contain the requested concept, state clearly:\n  "The agreement does not grant or define [topic]."\n- If no relevant information is found at all, state:\n  "The provided documents do not contain information regarding [topic]

llm answer: content='The agreement does not grant or define exclusivity rights. The only reference to “exclusive” pertains to jurisdiction (exclusive jurisdiction and venue for legal proceedings), not to any exclusivity rights between the parties. [Source: Section containing “this Agreement or the transactions contemplated hereby shall be instituted by a party in a Federal or state court sitting in the jurisdiction and venue of the other party, which shall be the exclusive jurisdiction and venue of said legal proceedings”]' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 422, 'prompt_tokens': 2346, 'total_tokens': 2768, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 438, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstr

In [115]:
answer
## expand just a 4 words and to not change the meaning of legal vocabulary

{'answer': 'The agreement does not grant or define exclusivity rights. The only reference to “exclusive” pertains to jurisdiction (exclusive jurisdiction and venue for legal proceedings), not to any exclusivity rights between the parties. [Source: Section containing “this Agreement or the transactions contemplated hereby shall be instituted by a party in a Federal or state court sitting in the jurisdiction and venue of the other party, which shall be the exclusive jurisdiction and venue of said legal proceedings”]',
 'sources': [{'filename': 'ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPORT AND MAINTENANCE AGREEMENT.PDF',
   'file_type': 'pdf',
   'page': 6,
   'contract_type': 'SUPPORT AND MAINTENANCE AGREEMENT',
   'agreement_date': '2005-04-01T00:00:00Z',
   'effective_date': '2005-04-01T00:00:00Z',
   'expiration_date': '2006-10-01T00:00:00Z',
   'agreement_date_human_display': '04/01/05',
   'effective_date_human_display': '04/01/05',
   'expiration_date_human_display': '10/01/06',


In [83]:
# from enum import Enum
# from pydantic import BaseModel, Field
# from typing import Optional
# ## enum

# class ConfidenceLevel(str, Enum):
#     HIGH   = "high"    
#     MEDIUM = "medium" 
#     LOW    = "low"

# class QueryIntent(str, Enum):
#     EXPIRY_DATE      = "expiry_date"       
#     TERMINATION      = "termination"       
#     LIABILITY        = "liability"         
#     PAYMENT          = "payment"           
#     GOVERNING_LAW    = "governing_law"     
#     AUTO_RENEWAL     = "auto_renewal"      
#     PARTIES          = "parties"           
#     CONFIDENTIALITY  = "confidentiality"  
#     GENERAL          = "general"           


# ## Cotation schema

# class Citation:
#     """
#     Points the user to the exact source of the answer.
#     Every answer MUST have at least one citation.
#     """
#     file_name: str = Field()
#     relevant_quote: Optional[str] = Field(
#         default=None,
#         description="Short verbatim quote (max 100 chars) from the contract"
#     )


# ## answer schema

 
# class ContractAnswer(BaseModel):
#     """
#     Structured response returned by the LLM for every contract query.

#     The LLM is instructed to populate all fields.
#     Guardrails validates this before it reaches the API.
#     """

#     answer: str = Field()

#     confidence: ConfidenceLevel = Field(
#         description="How confident the LLM is based on the retrieved context"
#     )

#     citations: list[Citation] = Field(
#         description="Source documents that support this answer (min 1 required)"
#     )

#     intent: QueryIntent = Field(
#         default=QueryIntent.GENERAL,
#         description="Detected intent of the user's query"
#     )

#     # Extracted structured data (when applicable)
#     expiry_date: Optional[str] = Field(
#         default=None,
#         description="Extracted expiry date if query is about contract end date"
#     )
#     contract_value: Optional[str] = Field(
#         default=None,
#         description="Extracted contract value if mentioned in answer"
#     )
#     notice_period_days: Optional[int] = Field(
#         default=None,
#         description="Termination notice period in days if mentioned"
#     )
#     metadata:dict

#     class Config:
#         use_enum_values = True


In [84]:
# import pandas as pd
# def clean_date_human_display(raw_data:str)->str:
#     # 1. Clean the string: Remove brackets and whitespace
#     raw_date = str(raw_data).replace("[", "").replace("]", "").strip()
#     # 2. Convert to a datetime object
#     date_obj = pd.to_datetime(raw_date, format='mixed', errors='coerce')
#     # 3. Format it back to your desired string style (MM/DD/YY)
#     if pd.notnull(date_obj):
#         if date_obj.year < 1900:
#             return f"{date_obj.month}/{date_obj.day}/{date_obj.year}"
#         # %m/%d/%y gives you 07/19/12. 
#         clean_date_str = date_obj.strftime('%m/%d/%y')
#     else:
#         clean_date_str = "Unknown" # Or keep it as None/nan
#     return clean_date_str

# ## clean date for qdrant database
# def clean_date_iso(raw_data:str):
#     # 1. Clean the string: Remove brackets and whitespace
#     raw_date = str(raw_data).replace("[", "").replace("]", "").strip()
#     # 2. Convert to a datetime object
#     date_obj = pd.to_datetime(raw_date, format='mixed', errors='coerce')
#     # 3. Format it back to ISO format (YYYY-MM-DD)
#     if pd.notnull(date_obj):
#         if date_obj.year < 1900:
#             return f"{date_obj.month}/{date_obj.day}/{date_obj.year}"
#         clean_date_str = date_obj.strftime('%Y-%m-%dT%H:%M:%SZ')
#     else:
#         clean_date_str = "Unknown" 
#     return clean_date_str

# def clear_nan(raw_value):
#     if pd.isna(raw_value):
#         return "Unknown"
#     else:
#         return raw_value
    
# def clear_brackets_and_nan(raw_value):
#     if pd.isna(raw_value):
#         return "Unknown"
#     else:
#         new_value =  str(raw_value).replace("[", "").replace("]", "").strip()
#         if new_value == "":
#             return "Unknown"
#         return new_value



In [85]:
# clean_date_iso(raw_data=" [2024-07-19] ")